# 🏭 Industrial Defect Detection System Using Computer Vision & Deep Learning (YOLO)
**Course**: Computer Vision CSE Assignment  
**Domain**: Automated High-Speed Industrial Manufacturing Inspection (Hot-Rolled Steel Strip Surface Defects)  
**Architecture**: Ultralytics YOLO (Transfer Learning) + Traditional Computer Vision Baseline  

---
### End-to-End Pipeline Stages:
1. **Environment Setup & GPU Verification**
2. **Dataset Acquisition & YOLO Format Annotation (NEU-DET Benchmark)**
3. **Industrial Preprocessing & Enhancement (CLAHE, Bilateral Denoising)**
4. **Deep Learning Model Training (YOLO Transfer Learning)**
5. **Quantitative Performance Evaluation (mAP@50, Precision, Recall, F1-Score)**
6. **Defect Prediction & Bounding Box Localization**
7. **Comparative Engineering Analysis: Traditional CV vs. Deep Learning**

## 1. Environment Setup & GPU Acceleration Check

In [ ]:
# Install required packages
!pip install -q ultralytics opencv-python matplotlib pandas seaborn pyyaml pillow tqdm

import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Accelerated: YES - {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running on CPU (For Colab, enable GPU under Runtime -> Change runtime type -> T4 GPU)")

## 2. Dataset Preparation (NEU Surface Defect Database - NEU-DET)
Prepares 6 manufacturing defect categories:
- `0: crazing` (micro-crack webs)
- `1: inclusion` (foreign slag particles)
- `2: patches` (oxide / discoloration regions)
- `3: pitted_surface` (cluster of micro-pits)
- `4: rolled-in_scale` (pressed rolling scale)
- `5: scratches` (mechanical surface grooves)

In [ ]:
import os
from pathlib import Path

# Create project directory structure
for split in ["train", "val", "test"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)
os.makedirs("results/graphs", exist_ok=True)
os.makedirs("results/predictions", exist_ok=True)
os.makedirs("results/sample_outputs", exist_ok=True)

# Write data.yaml configuration
data_yaml_content = """
path: dataset
train: images/train
val: images/val
test: images/test

nc: 6
names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches
"""
with open("data.yaml", "w") as f:
    f.write(data_yaml_content.strip())

# Populate dataset (generate high-fidelity annotated benchmark samples)
from src.prepare_dataset import setup_dataset
setup_dataset(force_recreate=True)

## 3. Industrial Image Preprocessing & Contrast Enhancement (CLAHE)
Compensates for factory environmental variations (uneven lighting, oil-film reflections, sensor grain noise).

In [ ]:
import matplotlib.pyplot as plt
import cv2
from src.preprocess import generate_preprocessing_visual_report

sample_train_img = list(Path("dataset/images/train").glob("*.jpg"))[0]
report_path = "results/graphs/preprocessing_pipeline_stages.png"
generate_preprocessing_visual_report(str(sample_train_img), report_path)

# Display visual report
img_rep = cv2.imread(report_path)
plt.figure(figsize=(16, 5))
plt.imshow(cv2.cvtColor(img_rep, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Multi-Stage Industrial Preprocessing Sequence", fontweight='bold')
plt.show()

## 4. Train Deep Learning Defect Detector (Ultralytics YOLO)
Trains lightweight YOLO architecture with transfer learning from pretrained weights.

In [ ]:
from ultralytics import YOLO

# Load pretrained model weights
model = YOLO("yolo11n.pt")

# Execute training
results = model.train(
    data="data.yaml",
    epochs=30,
    batch=16 if torch.cuda.is_available() else 4,
    imgsz=640,
    lr0=0.01,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/detect",
    name="train",
    plots=True,
    save=True,
    exist_ok=True
)

## 5. Quantitative Model Evaluation on Test Dataset
Evaluates Precision, Recall, F1-Score, mAP@50, and mAP@50-95 across all 6 defect categories.

In [ ]:
from src.evaluate import evaluate_defect_detector

eval_results = evaluate_defect_detector(model_path="runs/detect/train/weights/best.pt", split="test")

## 6. Batch Inference & Defect Localization
Runs inference on test images, renders bounding boxes, class labels, and quality inspection verdicts (PASS / REJECT).

In [ ]:
from src.predict import predict_batch
import glob

# Run batch prediction on test set
predict_batch(source_dir="dataset/images/test", model_path="runs/detect/train/weights/best.pt", conf_thresh=0.25)

# Display sample output detections in a 2x3 grid
sample_files = glob.glob("results/sample_outputs/*.jpg")[:6]
if sample_files:
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    for ax, sf in zip(axes.flatten(), sample_files):
        im = cv2.imread(sf)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(Path(sf).name, fontsize=10, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Comparative Engineering Analysis: Traditional CV vs. YOLO Deep Learning

In [ ]:
from src.traditional_cv import compare_traditional_vs_deep_learning

test_img = list(Path("dataset/images/test").glob("*.jpg"))[0]
comp_chart = "results/graphs/traditional_vs_yolo_comparison.png"
compare_traditional_vs_deep_learning(str(test_img), comp_chart)

# Display comparison image
im_comp = cv2.imread(comp_chart)
plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(im_comp, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Traditional Computer Vision vs. Deep Learning (YOLO) Pipeline Comparison", fontweight='bold')
plt.show()